# Projeto 3 — Variational Autoencoder (VAE)

In [6]:
import os
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from ucimlrepo import fetch_ucirepo

DATA_DIR = Path("../data/raw").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

## Dataset 1: Dry Bean
[Fonte: UCI Machine Learning Repository, ID 602](https://archive.ics.uci.edu/dataset/602/dry+bean+dataset). São 13.611 amostras, 16 atributos numéricos e 7 classes de feijão.

In [7]:
dry_bean_path = DATA_DIR / "dry_bean.csv"
if dry_bean_path.exists():
    dry_bean_df = pd.read_csv(dry_bean_path)
else:
    dry_bean = fetch_ucirepo(id=602)
    dry_bean_df = pd.concat(
        [dry_bean.data.features, dry_bean.data.targets], axis=1
    )
    dry_bean_df.to_csv(dry_bean_path, index=False)

X_dry_bean = dry_bean_df.drop(columns="Class").copy()
y_dry_bean = dry_bean_df["Class"].copy()

print(f"Dry Bean: {X_dry_bean.shape[0]} amostras, {X_dry_bean.shape[1]} atributos")
display(X_dry_bean.head())
display(y_dry_bean.value_counts().sort_index().rename("amostras").to_frame())


Dry Bean: 13611 amostras, 16 atributos


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166


,amostras
Class,
BARBUNYA,1322
BOMBAY,522
CALI,1630
DERMASON,3546
HOROZ,1928
SEKER,2027
SIRA,2636


## Pré-processamento do dataset Dry Bean

Verificamos ausentes e retiramos duplicatas. Dividimos em treino, validação e teste com estratificação e seed fixa. A tabela compara as porcentagens de cada classe. Padronizamos os atributos usando somente o treino.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#Conferir ausentes e retirar duplicatas antes da divisão.
print("Valores ausentes no Dry Bean:", dry_bean_df.isna().sum().sum())
dry_clean = dry_bean_df.drop_duplicates()
print("Duplicatas removidas:", len(dry_bean_df) - len(dry_clean))

# Dry Bean: 70% treino, 15% validação e 15% teste.
X_dry_train, X_dry_test, y_dry_train, y_dry_test = train_test_split(
    dry_clean.drop(columns="Class"), dry_clean["Class"],
    test_size=0.15, stratify=dry_clean["Class"], random_state=SEED
)
X_dry_train, X_dry_val, y_dry_train, y_dry_val = train_test_split(
    X_dry_train, y_dry_train,
    test_size=0.15 / 0.85, stratify=y_dry_train, random_state=SEED
)

#Padronização dos dados (StandardScaler) para o Dry Bean.
scaler_dry = StandardScaler()
X_dry_train = scaler_dry.fit_transform(X_dry_train).astype("float32")
X_dry_val = scaler_dry.transform(X_dry_val).astype("float32")
X_dry_test = scaler_dry.transform(X_dry_test).astype("float32")


def conferir_classes(y_train, y_val, y_test):
    conjuntos = {"treino": y_train, "validação": y_val, "teste": y_test}
    print("Amostras:", {nome: len(y) for nome, y in conjuntos.items()})
    display(pd.DataFrame({
        nome: pd.Series(y).value_counts(normalize=True) * 100
        for nome, y in conjuntos.items()
    }).sort_index().round(2).rename_axis("classe (%)"))


conferir_classes(y_dry_train, y_dry_val, y_dry_test)

Valores ausentes no Dry Bean: 0
Duplicatas removidas: 68
Amostras: {'treino': 9479, 'validação': 2032, 'teste': 2032}


,treino,validação,teste
classe (%),,,
BARBUNYA,9.77,9.74,9.74
BOMBAY,3.86,3.84,3.84
CALI,12.03,12.06,12.06
DERMASON,26.18,26.18,26.18
HOROZ,13.74,13.73,13.73
SEKER,14.97,14.96,14.96
SIRA,19.45,19.49,19.49


## Dataset 2: MNIST
[Fonte: MNIST pelo Keras](https://keras.io/api/datasets/mnist/). São imagens de dígitos de 0 a 9, em tons de cinza, com 28 × 28 pixels. Preservamos a divisão oficial: 60.000 imagens para treino e 10.000 para teste.

In [9]:
(X_mnist_train, y_mnist_train), (X_mnist_test, y_mnist_test) = (
    keras.datasets.mnist.load_data()
)

print(f"MNIST treino: {X_mnist_train.shape}, rótulos: {y_mnist_train.shape}")
print(f"MNIST teste: {X_mnist_test.shape}, rótulos: {y_mnist_test.shape}")
display(pd.DataFrame({
    "treino": pd.Series(y_mnist_train).value_counts().sort_index(),
    "teste": pd.Series(y_mnist_test).value_counts().sort_index(),
}).rename_axis("dígito"))


MNIST treino: (60000, 28, 28), rótulos: (60000,)
MNIST teste: (10000, 28, 28), rótulos: (10000,)


,treino,teste
dígito,,
0,5923,980
1,6742,1135
2,5958,1032
3,6131,1010
4,5842,982
5,5421,892
6,5918,958
7,6265,1028
8,5851,974


## Pré-processamento do MNIST

Preservamos as 10.000 imagens de teste oficiais e dividimos as 60.000 restantes em 50.000 de treino e 10.000 de validação. Normalizamos os pixels para [0, 1] em float32 e mantemos o formato 28 × 28.

In [10]:
#Separar 10.000 imagens para validação e manter o teste oficial intacto.
X_mnist_train_split, X_mnist_val, y_mnist_train_split, y_mnist_val = train_test_split(
    X_mnist_train, y_mnist_train,
    test_size=1/6, stratify=y_mnist_train, random_state=SEED
)

#Normalizar os dados do MNIST para o intervalo [0, 1].
X_mnist_train_norm = X_mnist_train_split.astype("float32") / 255.0
X_mnist_val_norm = X_mnist_val.astype("float32") / 255.0
X_mnist_test_norm = X_mnist_test.astype("float32") / 255.0

conferir_classes(y_mnist_train_split, y_mnist_val, y_mnist_test)

Amostras: {'treino': 50000, 'validação': 10000, 'teste': 10000}


,treino,validação,teste
classe (%),,,
0,9.87,9.87,9.80
1,11.24,11.24,11.35
2,9.93,9.93,10.32
3,10.22,10.22,10.10
4,9.74,9.74,9.82
5,9.04,9.03,8.92
6,9.86,9.86,9.58
7,10.44,10.44,10.28
8,9.75,9.75,9.74


## Implementação do VAE

- **Encoder:** resume os dados em uma representação menor, chamada espaço latente.
- **Amostragem:** adiciona uma variação aleatória ao resumo produzido pelo encoder.
- **Decoder:** usa esse resumo para tentar reconstruir os dados originais.
- **Custo:** mede o erro da reconstrução e ajuda a organizar o espaço latente (termo KL).

O **MSE (erro quadrático médio)** mede a distância entre os valores originais e os reconstruídos, penalizando mais os erros grandes. Ele serve tanto para reconstruir as medidas do Dry Bean quanto as intensidades dos pixels do MNIST.

Usamos, portanto, o mesmo critério de reconstrução nos dois datasets. A saída do decoder continua **linear no Dry Bean**, para permitir valores negativos após a padronização, e **sigmoid no MNIST**, para manter os pixels entre 0 e 1. 

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, tipo):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tipo = tipo

    def call(self, inputs, training=False):

        #O encoder produz a média e o logaritmo da variância do resumo.
        z_mean, z_log_var = self.encoder(inputs, training=training)

        #Amostragem: combina a média, o desvio e um ruído aleatório.
        ruido = tf.random.normal(tf.shape(z_mean))
        z = z_mean + tf.exp(0.5 * z_log_var) * ruido

        #O decoder tenta recuperar a entrada a partir do resumo z.
        reconstruido = self.decoder(z, training=training)

        #Coloca os dados em vetores para comparar entrada e reconstrução.
        original = tf.reshape(inputs, (tf.shape(inputs)[0], -1))
        saida = tf.reshape(reconstruido, (tf.shape(inputs)[0], -1))

        #O erro quadrático médio (MSE) mede a diferença entre a entrada e a reconstrução.
        mse = tf.reduce_mean(tf.square(original - saida), axis=1)
        recon_loss = mse * tf.cast(tf.shape(original)[1], mse.dtype)

        #KL: incentiva o espaço latente a seguir uma normal padrão.
        kl_loss = -0.5 * tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)

        self.add_loss(tf.reduce_mean(recon_loss + kl_loss))
        return reconstruido


def criar_vae(input_shape, latent_dim=2, camadas=(64, 32), tipo="tabular", learning_rate=1e-3):

    #Cria a rede.
    if tipo not in ("tabular", "imagem"):
        raise ValueError("tipo deve ser 'tabular' ou 'imagem'.")
    input_shape = tuple(input_shape)

    #Encoder: transforma a entrada em um resumo menor.
    entrada = keras.Input(shape=input_shape)
    x = keras.layers.Flatten()(entrada)
    for unidades in camadas:
        x = keras.layers.Dense(unidades, activation="relu")(x)
    z_mean = keras.layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = keras.layers.Dense(latent_dim, name="z_log_var")(x)
    encoder = keras.Model(entrada, [z_mean, z_log_var], name="encoder")

    #Decoder: percorre as camadas na ordem inversa e reconstrói a entrada.
    entrada_z = keras.Input(shape=(latent_dim,))
    x = entrada_z
    for unidades in reversed(camadas):
        x = keras.layers.Dense(unidades, activation="relu")(x)

    #Sigmoid limita pixels a [0, 1], já o linear permite valores negativos no Dry Bean.
    ativacao = "sigmoid" if tipo == "imagem" else "linear"
    x = keras.layers.Dense(int(np.prod(input_shape)), activation=ativacao)(x)

    # Recupera o formato original: 16 atributos ou imagem 28 × 28.
    saida = keras.layers.Reshape(input_shape)(x)
    decoder = keras.Model(entrada_z, saida, name="decoder")

    #Junta as partes e escolhe o otimizador que ajustará os pesos.
    vae = VAE(encoder, decoder, tipo)
    vae.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate))
    return vae


def treinar_vae(vae, X_train, X_val, epochs=100, batch_size=128, patience=10, nome="vae"):

    #Para após patience épocas sem melhora na validação e recupera os melhores pesos.
    parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)

    pasta = DATA_DIR.parents[1] / "results" / "historicos"
    pasta.mkdir(parents=True, exist_ok=True)
    arquivo = pasta / f"{nome}_{pd.Timestamp.now():%Y%m%d_%H%M%S_%f}.csv"
    registro = keras.callbacks.CSVLogger(str(arquivo))

    #Aprende com o treino e acompanha a validação, sem usar rótulos de classes.
    historico = vae.fit(
        X_train, validation_data=(X_val,), epochs=epochs, batch_size=batch_size,
        callbacks=[parada, registro], shuffle=True
    )
    print(f"Perdas salvas em: {arquivo}")
    return historico